# Plant Disease Classifier — Colab Training

Run these cells top to bottom on a GPU runtime (Runtime > Change runtime type > GPU).

1. Clone the repo
2. Install the few extra dependencies (Colab's preinstalled PyTorch is used)
3. Download the dataset via the Kaggle API
4. Train (writes to `results/colab_run/`, never over the committed `results/best_model.pt`)
5. Evaluate

> Colab's PyTorch is newer than the version pinned in `requirements.txt` (which targets Python 3.11).
> Checkpoints saved here still load with the pinned version, because they contain only tensors and plain metadata.

In [ ]:
!git clone https://github.com/TakshChauhan/Plant-Disease-Classifier.git
%cd Plant-Disease-Classifier

In [ ]:
# torch/torchvision/numpy/matplotlib/scikit-learn already ship with Colab; don't reinstall the pinned torch
!pip install -q scikit-learn matplotlib tqdm kaggle
import torch, torchvision, sys
print(sys.version.split()[0], torch.__version__, torchvision.__version__, "CUDA:", torch.cuda.is_available())

In [ ]:
# Upload your kaggle.json (Kaggle account -> Settings -> Create New Token) when prompted.
# It is copied to ~/.kaggle and then deleted from the working directory; never commit it.
from google.colab import files
files.upload()  # select kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p data --unzip -q

In [ ]:
# The Kaggle zip is nested; find the folder that directly contains train/ and valid/
from pathlib import Path
candidates = [p for p in Path("data").rglob("train") if (p.parent / "valid").is_dir()]
assert candidates, "Could not find train/ and valid/ under data/"
DATA_DIR = str(candidates[0].parent)
print("DATA_DIR =", DATA_DIR)

In [ ]:
!python src/train.py --data-dir "$DATA_DIR" --model resnet18 --freeze-epochs 3 --finetune-epochs 7 --batch-size 32 --output-dir results/colab_run

In [ ]:
!python src/evaluate.py --data-dir "$DATA_DIR" --split valid --checkpoint results/colab_run/best_model.pt --output-dir results/colab_run/evaluation

In [ ]:
from IPython.display import Image, display
display(Image('results/colab_run/training_curves.png'))
display(Image('results/colab_run/evaluation/confusion_matrix.png'))
print(open('results/colab_run/evaluation/metrics.json').read())

**Interpreting the numbers.** `valid/` is used both for early stopping / checkpoint selection and for this
evaluation, so the reported accuracy is optimistic. The dataset is offline-augmented PlantVillage, so near-duplicate
images of the same leaf can appear in both `train/` and `valid/`. Treat validation accuracy as an in-distribution
sanity check, not as evidence of real-world performance (see the README's *Limitations* section).